# NO2 - Regresi Linear & Logistik
**Pendamping S06 - Kecerdasan Buatan & ML dasar**

Pola kerja: **DO → OBSERVE → INTERPRET → EXPLAIN**. Jalankan sel secara berurutan, catat evidence, lalu tuliskan interpretasi Anda.

**KELOMPOK A**



*   Dica Damar Bintang (261635003)
*   Nadhilah Salsabila Rahmadhani (261636001)



 ### Bagian A — Linear Regression

  ### Eksperimen 1

In [8]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, confusion_matrix, classification_report
from sklearn.datasets import load_breast_cancer

rng = np.random.default_rng(42)
X = rng.uniform(20, 120, size=(200,1))
y = 50 + 1.8*X[:,0] + rng.normal(0, 18, size=200)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.25, random_state=42)
lin = LinearRegression().fit(X_train, y_train)
pred = lin.predict(X_test)
print('intercept=', lin.intercept_, 'coef=', lin.coef_[0])
print('MAE=', mean_absolute_error(y_test,pred))
print('RMSE=', mean_squared_error(y_test,pred)**0.5)

intercept= 39.9784077381326 coef= 1.9237076136073443
MAE= 13.709271226328251
RMSE= 17.364824347784438


In [9]:
resid = y_test - pred
idx = np.argsort(np.abs(resid))[-3:]
pd.DataFrame({
    'aktual': y_test[idx],
    'prediksi': pred[idx],
    'residual': resid[idx]
})

,aktual,prediksi,residual
0,199.721397,166.734537,32.986860
1,137.384165,98.344268,39.039897
2,265.371721,221.723023,43.648698


Interpretasikan koefisien tanpa mengklaim kausalitas. Cari 3 residual terbesar.

Koefisien sekitar 1,924 berarti kenaikan satu unit X berkaitan dengan kenaikan prediksi Y sekitar 1,924 unit pada data ini. Intercept adalah prediksi saat X = 0, di luar rentang X yang diamati. Ini menggambarkan hubungan pada model, bukan bukti sebab-akibat. MAE (13,709) menunjukkan besar kesalahan absolut rata-rata; RMSE (17,365) lebih sensitif terhadap kesalahan yang besar.

### Bagian B — Logistic Regression

### Eksperimen 2 — Logistic Regression dengan Variasi Threshold

In [10]:
data = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=.25, random_state=42, stratify=data.target
)
log = LogisticRegression(max_iter=5000).fit(X_train, y_train)
proba = log.predict_proba(X_test)[:,1]

for t in [0.3, 0.5, 0.7]:
    pred_t = (proba >= t).astype(int)
    print('\nthreshold', t)
    print(confusion_matrix(y_test,pred_t))
    print(classification_report(y_test,pred_t, digits=3))


threshold 0.3
[[46  7]
 [ 1 89]]
              precision    recall  f1-score   support

           0      0.979     0.868     0.920        53
           1      0.927     0.989     0.957        90

    accuracy                          0.944       143
   macro avg      0.953     0.928     0.938       143
weighted avg      0.946     0.944     0.943       143


threshold 0.5
[[48  5]
 [ 1 89]]
              precision    recall  f1-score   support

           0      0.980     0.906     0.941        53
           1      0.947     0.989     0.967        90

    accuracy                          0.958       143
   macro avg      0.963     0.947     0.954       143
weighted avg      0.959     0.958     0.958       143


threshold 0.7
[[49  4]
 [ 4 86]]
              precision    recall  f1-score   support

           0      0.925     0.925     0.925        53
           1      0.956     0.956     0.956        90

    accuracy                          0.944       143
   macro avg      0.940   

Bandingkan threshold 0.3, 0.5, 0.7. Apa yang berubah pada false positive dan false negative?

Threshold 0,3: Model menghasilkan 7 FP (False Positive) dan 1 FN (False Negative). FP berarti data ganas salah diprediksi jinak, sedangkan FN berarti data jinak salah diprediksi ganas.

Threshold 0,5: Akurasi mencapai 95,8% dengan 5 FP dan 1 FN. Hasil ini merupakan yang terbaik dari ketiga percobaan.

Threshold 0,7: FP turun menjadi 4, tetapi FN naik menjadi 4. Artinya, jumlah kesalahan dalam mengklasifikasikan kedua jenis data mengalami perubahan.

**Mengapa hasil threshold bisa berbeda?**

Logistic Regression menghasilkan probabilitas yang digunakan untuk menentukan kelas melalui threshold. Jika threshold diubah, hasil klasifikasi dan jumlah kesalahan juga dapat berubah.

### Eksperimen 3 — Pengaruh Tingkat Noise terhadap Performa Linear Regression

In [11]:
# DO — Menguji pengaruh tiga tingkat noise pada dataset sintetis.
# Seed 2026 berbeda dari seed 42 yang digunakan pada Eksperimen 1.
rng_exp3 = np.random.default_rng(2026)

# X dan pola noise dasar dibuat satu kali agar perbandingan antar-level noise adil.
X_exp3 = rng_exp3.uniform(20, 120, size=(200, 1))
base_noise = rng_exp3.normal(0, 1, size=200)

noise_levels = [5, 18, 30]
results = []

for noise in noise_levels:
    # Fungsi dasar: y = 50 + 1.8X, dengan noise yang skalanya diubah.
    y_exp3 = 50 + 1.8 * X_exp3[:, 0] + noise * base_noise

    X_train_exp3, X_test_exp3, y_train_exp3, y_test_exp3 = train_test_split(
        X_exp3, y_exp3, test_size=0.25, random_state=42
    )

    model_exp3 = LinearRegression().fit(X_train_exp3, y_train_exp3)
    pred_exp3 = model_exp3.predict(X_test_exp3)

    results.append({
        "Tingkat Noise (Std)": noise,
        "Intercept": model_exp3.intercept_,
        "Koefisien": model_exp3.coef_[0],
        "MAE": mean_absolute_error(y_test_exp3, pred_exp3),
        "RMSE": mean_squared_error(y_test_exp3, pred_exp3) ** 0.5
    })

df_exp3 = pd.DataFrame(results)

# OBSERVE — Tabel membandingkan parameter model dan error prediksi.
df_exp3.round(3)

,Tingkat Noise (Std),Intercept,Koefisien,MAE,RMSE
0,5,52.252,1.770,3.677,4.885
1,18,58.108,1.692,13.238,17.586
2,30,63.513,1.620,22.063,29.311


OBSERVE — Hasil Eksperimen 3

Tabel pada sel sebelumnya menampilkan hasil untuk noise 5, 18, dan 30. Nilai yang ditampilkan dibulatkan menjadi tiga angka desimal agar mudah dibandingkan.

### INTERPRET — Interpretasi hasil

1. **Pengaruh noise terhadap error prediksi:**
   - Pada noise 5, MAE = **3,677** dan RMSE = **4,885**.
   - Pada noise 18, MAE = **13,238** dan RMSE = **17,586**.
   - Pada noise 30, MAE = **22,063** dan RMSE = **29,311**.
   - MAE dan RMSE meningkat ketika tingkat noise dinaikkan. Artinya, prediksi model menjadi semakin jauh dari nilai aktual karena data mengandung variasi acak yang lebih besar.

2. **Pengaruh noise terhadap koefisien:**
   - Koefisien yang diperoleh berturut-turut adalah **1,770**, **1,692**, dan **1,620**.
   - Fungsi pembentuk data menggunakan koefisien 1,8. Pada percobaan ini, estimasi koefisien semakin menjauh dari 1,8 ketika noise meningkat.
   - Perubahan koefisien dan intercept dapat terjadi karena model dilatih dari sampel terbatas yang mengandung noise acak.

### EXPLAIN — Kesimpulan

Dengan dataset sintetis, seed 2026, dan pembagian data yang sama, peningkatan standar deviasi noise dari 5 menjadi 30 diikuti kenaikan MAE dari 3,677 menjadi 22,063 serta RMSE dari 4,885 menjadi 29,311. Ini menunjukkan bahwa noise yang lebih tinggi membuat pola hubungan linear lebih sulit diperkirakan secara akurat. Hasil ini berlaku untuk data dan pengaturan eksperimen yang diuji, bukan berarti setiap dataset nyata akan menunjukkan angka atau pola parameter yang persis sama.

## Evidence Log

| Eksperimen | Fokus | Metric / Output | Interpretasi |
|---|---|---|---|
| EXP-01 | Regresi Linear baseline | Koefisien: 1,924; MAE: 13,709; RMSE: 17,365 | Model mampu mengenali pola X dan Y, tetapi masih terdapat kesalahan prediksi. |
| EXP-02 | Logistic Regression dengan threshold 0,3 / 0,5 / 0,7 | FP: 7 / 5 / 4; FN: 1 / 1 / 4; akurasi: 0,944 / 0,958 / 0,944 | Threshold 0,5 menghasilkan akurasi tertinggi sebesar 0,958. |
| EXP-03 | Pengaruh tingkat noise (std = 5, 18, 30) terhadap Linear Regression; seed = 2026 | Noise 5: koefisien 1,770; MAE 3,677; RMSE 4,885. Noise 18: koefisien 1,692; MAE 13,238; RMSE 17,586. Noise 30: koefisien 1,620; MAE 22,063; RMSE 29,311. | MAE dan RMSE meningkat pada noise yang lebih tinggi. Estimasi koefisien pada percobaan ini semakin menjauh dari nilai pembentuk data, yaitu 1,8. |

## Refleksi akhir

1. **Apa hasil utama yang Anda amati?**

   EXP-01 menunjukkan Regresi Linear baseline mampu mengenali pola data. EXP-02 menunjukkan threshold 0,5 pada Logistic Regression memiliki akurasi tertinggi sebesar 0,958. EXP-03 menunjukkan bahwa ketika tingkat noise dinaikkan dari 5 ke 30, MAE meningkat dari 3,677 menjadi 22,063 dan RMSE meningkat dari 4,885 menjadi 29,311. Pada percobaan ini, estimasi koefisien juga berubah dari 1,770 menjadi 1,620, menjauh dari koefisien pembentuk data 1,8.

2. **Mengapa hasil tersebut terjadi menurut konsep di Main Deck?**

   Regresi Linear memprediksi nilai berdasarkan hubungan linier antara X dan Y. Noise menambahkan variasi acak yang tidak dijelaskan oleh X. Semakin besar variasi acak tersebut, semakin besar kemungkinan nilai prediksi berbeda dari nilai aktual sehingga MAE dan RMSE meningkat. Sementara itu, Logistic Regression menggunakan threshold probabilitas untuk menentukan kelas. Perubahan threshold dapat mengubah trade-off antara False Positive dan False Negative.

3. **Apa limitation eksperimen ini?**

   Eksperimen 3 menggunakan data sintetis satu fitur (1D), 200 sampel, dan noise Gaussian. Hasilnya dipengaruhi oleh seed dan pembagian data yang digunakan. Data nyata dapat memiliki pola yang lebih kompleks, noise yang tidak berdistribusi normal, atau beberapa fitur yang saling berkaitan.

4. **Eksperimen apa yang layak dilakukan berikutnya?**

   Mengulang pengujian dengan beberapa random seed untuk melihat kestabilan hasil, atau membandingkan Linear Regression biasa dengan Ridge/Lasso pada tingkat noise yang sama.